In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
os.environ['JAX_PLATFORMS'] = 'cpu'
import jax
import jax.numpy as jnp
import jax.random as random
import optax
from tqdm import tqdm
from utils import DataLoader
from flax import nnx
from flax.nnx.training.metrics import Metric, Average

In [3]:
import jax
import jax.numpy as jnp
import jax.random as random
import optax
from tqdm import tqdm
from utils import DataLoader
from flax import nnx
from flax.nnx.training.metrics import Metric, Average

# DAJE

In [31]:
#GENERAZIONE DEL DATASET
key = random.key(0)
f_to_learn = lambda mu, l, k, x: (mu+l+k)*x
N = 100
key, subkey = random.split(key) 
x = random.uniform(subkey, (N,), minval=-10, maxval=10)
key, subkey = random.split(key) 
mu = random.uniform(subkey, (N,), minval=-10, maxval=10)
key, subkey = random.split(key) 
l = random.uniform(subkey, (N,), minval=-10, maxval=10)
key, subkey = random.split(key) 
k = random.uniform(subkey, (N,), minval=-10, maxval=10)
y = f_to_learn(mu, l, k, x) # così generiamo artificialmente un dataset di N punti
X = jnp.stack([mu, l, k, x], axis=1)

# DIVISIONE DEL DATASET IN TRAIN E TEST E CREAZIONE DEI DATALOADER
split_idx = int(N * 0.8)
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]
train_dataloader = DataLoader(X_train, y_train, batch_size=1, shuffle=True)
test_dataloader = DataLoader(X_test, y_test, batch_size=1, shuffle=False) #CAMBIARE BATCHSIZE

In [35]:
hypernetwork = nnx.Linear(3, 2, rngs=nnx.Rngs(0))
targetnetwork = nnx.Linear(1, 1, rngs=nnx.Rngs(0))

In [22]:
nnx.display(targetnetwork)
graphdef, state = nnx.split(targetnetwork)
nnx.display(graphdef, state)

In [ ]:
print(state)
w = jnp.array([1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0])

def assign_params(state, params):
    """
    Assigns the parameters from an array to the state.
    """
    params = params.squeeze()
    flat_state = nnx.to_flat_state(state)
  
    # TODO: Check if the number of parameters matches
    
    i=0
    for (key, param) in flat_state:
        param.value = params[i: i + param.value.size].reshape(param.value.shape)
        i += param.value.size   

#assign_params = nnx.vmap(assign_params) NOPE :(

#Esempio
assign_params(state, w)
print(state)

State({
  'bias': VariableState( # 1 (4 B)
    type=Param,
    value=Array([1], dtype=int32)
  ),
  'kernel': VariableState( # 1 (4 B)
    type=Param,
    value=Array([[2]], dtype=int32)
  )
})
State({
  'bias': VariableState( # 1 (4 B)
    type=Param,
    value=Array([1.], dtype=float32)
  ),
  'kernel': VariableState( # 1 (4 B)
    type=Param,
    value=Array([[2.]], dtype=float32)
  )
})


In [66]:
w = jnp.array([[1.0, 2.0], [3.0, 4.0], [5.0, 6.0], [7.0, 8.0]])

def assign_params(params: jax.Array):
    params = params.squeeze()

    # Clone the graphdef and reconstruct a model
    graphdef, state = nnx.split(targetnetwork)
    flat_state = nnx.to_flat_state(state)

    i = 0
    for (_, param) in flat_state:
        print(params[0:1])
        param.value = params[i: i + param.value.size].reshape(param.value.shape)
        i += param.value.size

assign_models = nnx.vmap(assign_params, in_axes=0)
assign_params(w[0,:])
print(state)

[1.]
[1.]
State({
  'bias': VariableState( # 1 (4 B)
    type=Param,
    value=Array([0.], dtype=float32)
  ),
  'kernel': VariableState( # 1 (4 B)
    type=Param,
    value=Array([[1.0701778]], dtype=float32)
  )
})


In [216]:
def train_step(hypernetwork, graphdef, state, hyperparams, x, y, optimizer):
    def loss_fn(hypernetwork, hyperparams, x, y):
        w = hypernetwork(hyperparams)
        assign_params(state, w)
        targetnetwork = nnx.merge(graphdef, state)
        pred = targetnetwork(x)
        loss = jnp.mean(optax.l2_loss(pred, y))
        return loss
    loss, grads = nnx.value_and_grad(loss_fn)(hypernetwork, hyperparams, x, y)
    optimizer.update(grads)
    return loss

train_step = nnx.jit(train_step)

In [ ]:
graphdef, state = nnx.split(targetnetwork)
optimizer = nnx.Optimizer(hypernetwork, optax.adam(learning_rate=1e-3))
for epoch in range(100):
    loss = 0.0
    for data, label in train_dataloader:
        hyperparams = data[:, :-1] # mu, l, k
        x = data[:, -1:] # x
        loss += train_step(hypernetwork, graphdef, state, hyperparams, x, label, optimizer)

    print(loss / len(train_dataloader))

4780.2124
4607.884
4413.4365
4216.7944
4023.0366
3833.927
3650.2114
3472.233
3300.136
3133.9563
2973.6682
2819.2085
2670.49
2527.415
2389.8708
2257.7432
2130.9114
2009.2561
1892.6539
1780.982
1674.1199
1571.9459
1474.34
1381.1827
1292.3574
1207.7458
1127.2335
1050.7057
978.0482
909.1489
843.8966
782.1796
723.8884
668.91296
617.14435
568.47424
522.79407
479.9961
439.97272
402.6171
367.82172
335.4803
305.48633
277.73383
252.11707
228.53091
206.87082
187.03288
168.91461
152.41388


# CELIC DRAPER

In [30]:
class MLP(nnx.Module):
    input_dim: int
    output_dim: int = 1
    hidden_dim: int = 8
    num_hidden_layers: int = 1

    def __init__(self, input_dim: int, output_dim: int, hidden_dim: int, num_hidden_layers: int, *, rngs: nnx.Rngs):
        self.input_dim = input_dim
        self.output_dim = output_dim
        self.hidden_dim = hidden_dim
        self.num_hidden_layers = num_hidden_layers
        if num_hidden_layers > 0:
            self.hidden_layers = [nnx.Linear(self.input_dim, self.hidden_dim, rngs=rngs)] + \
                [nnx.Linear(self.hidden_dim, self.hidden_dim, rngs=rngs) for _ in range(self.num_hidden_layers - 1)]
            self.linear = nnx.Linear(hidden_dim, output_dim, rngs=rngs)
        else:
            self.linear = nnx.Linear(self.input_dim, self.output_dim, rngs=rngs)

    def __call__(self, x: jax.Array):
        if self.num_hidden_layers > 0:
            for i in range(self.num_hidden_layers):
                x = self.hidden_layers[i](x)
                x = nnx.relu(x) 
            x = self.linear(x) 
        else:
            x = self.linear(x)
        return x

In [31]:
targetnetwork = MLP(input_dim = 1, output_dim = 1, hidden_dim = 8, num_hidden_layers = 0, rngs=nnx.Rngs(0))

In [44]:
targetnetwork = nnx.Linear(1, 1, rngs=nnx.Rngs(0))
graphdef, state = nnx.split(targetnetwork)

In [43]:
w = jnp.array([[1.0, 2.0], [3.0, 4.0], [5.0, 6.0], [7.0, 8.0]])

def assign_params(params: jax.Array):
    params = params.squeeze()

    # Clone the graphdef and reconstruct a model
    targetnetwork = nnx.Linear(1, 1, rngs=nnx.Rngs(0))
    graphdef, state = nnx.split(targetnetwork)
    flat_state = nnx.to_flat_state(state)

    new_state = {}
    i = 0
    for (key, param) in flat_state:
        param_size = param.value.size
        # Extract the right slice and reshape to match parameter shape
        new_value = params[i:i + param_size].reshape(param.value.shape)
        # Create new Variable with the new value
        new_state[key] = nnx.Variable(new_value)
        i += param_size
    
    # Reconstruct the complete model with new parameters
    new_model = nnx.merge(graphdef, nnx.from_flat_state(new_state))
    return new_model

new_model = nnx.vmap(assign_params, in_axes=0, out_axes=0)(w)
print(new_model)

Linear( # Variable: 8 (32 B)
  bias=Variable( # 4 (16 B)
    value=Array(shape=(4, 1), dtype=dtype('float32'))
  ),
  bias_init=<function zeros at 0x7ade2b336660>,
  dot_general=<function dot_general at 0x7ade2bbeab60>,
  dtype=None,
  in_features=1,
  kernel=Variable( # 4 (16 B)
    value=Array(shape=(4, 1, 1), dtype=dtype('float32'))
  ),
  kernel_init=<function variance_scaling.<locals>.init at 0x7ade2a2156c0>,
  out_features=1,
  param_dtype=float32,
  precision=None,
  promote_dtype=<function promote_dtype at 0x7ade2a215080>,
  use_bias=True
)


In [45]:
def get_state_from_params(params: jax.Array):
    params = params.squeeze()
    targetnetwork = nnx.Linear(1, 1, rngs=nnx.Rngs(0))
    graphdef, state = nnx.split(targetnetwork)
    flat_state = nnx.to_flat_state(state)

    new_state = {}
    i = 0
    for (key, param) in flat_state:
        param_size = param.value.size
        new_value = params[i:i + param_size].reshape(param.value.shape)
        new_state[key] = nnx.Variable(new_value)
        i += param_size

    return new_state  # graphdef is the same, can be reused

# Use vmap to build 4 different parameter states
states = nnx.vmap(get_state_from_params, in_axes=0, out_axes=0)(w)

# Rebuild 4 modules
models = [nnx.merge(graphdef, state) for state in states]

ValueError: Incorrect number of leaves, expected 2 leaves, but got 1.